# Lezione 5 — Pratica: da modellazione a valutazione

In questa pratica addestri il tuo albero decisionale con scikit-learn sul dataset di **57.691 ricette di cucina**, come lo hai visto nel notebook di teoria e nel [capitolo 5 del manuale](../manual/06_modelado_evaluacion_IT.md).

Lo stesso notebook ufficiale di IBM suggerisce questo esercizio: *"Feel free to select another subset of cuisines and build a decision tree of their recipes. You can select some European cuisines and build decision trees to explore the ingredients that differentiate those cuisines."* Questo è esattamente ciò che farai.

Due esercizi:

1. Costruire un albero decisionale con un sottoinsieme di **cucine europee** e valutarlo con una matrice di confusione.
2. Calcolare a mano l'accuracy globale del tuo albero con `accuracy_score`.

Prima di iniziare, esegui la cella sotto: carica e pulisce il dataset di ricette esattamente come nelle lezioni precedenti, così parti già da un `recipes` pulito e pronto all'uso.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import random
from sklearn import tree
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import itertools

recipes = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv"
)

column_names = recipes.columns.values
column_names[0] = "cuisine"
recipes.columns = column_names
recipes["cuisine"] = recipes["cuisine"].str.lower()

recipes.loc[recipes["cuisine"] == "korea", "cuisine"] = "korean"
recipes.loc[recipes["cuisine"] == "japan", "cuisine"] = "japanese"
recipes.loc[recipes["cuisine"] == "china", "cuisine"] = "chinese"
recipes.loc[recipes["cuisine"] == "thailand", "cuisine"] = "thai"
recipes.loc[recipes["cuisine"] == "india", "cuisine"] = "indian"

recipes_counts = recipes["cuisine"].value_counts()
cuisines_indices = recipes_counts > 50
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])
recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

# Verifica quali nomi di cucina europea hai disponibili nel dataset già pulito
print(sorted(recipes["cuisine"].unique()))
print("Dataset limpio y listo:", recipes.shape)

## Esercizio 1 — Albero decisionale con cucine europee

Il tuo compito è ripetere il flusso del notebook di teoria (addestramento, train/test, previsione, matrice di confusione), ma questa volta con un sottoinsieme di **cucine europee** invece che asiatiche e indiana.

**Suggerimenti:**
- Guarda la lista di cucine che hai stampato sopra e scegli tra 4 e 5 cucine europee presenti nel dataset (ad esempio `"french"`, `"italian"`, `"german"`, `"spanish_portuguese"` — usa esattamente i nomi come appaiono nella lista stampata, può darsi che qualcuno non esista e tu debba aggiustare la selezione).
- Il flusso è: filtrare `recipes` per quelle cucine → separare `ingredients` (colonne 1 in poi) e `cuisines` (colonna "cuisine") → dividere in train/test con `sample_n` ricette per cucina → addestrare un `DecisionTreeClassifier` con l'insieme di addestramento → prevedere con `.predict()` sull'insieme di test → disegnare la matrice di confusione.
- Usa `random.seed(1234)` come nel notebook di teoria affinché il tuo risultato sia riproducibile.
- Se qualche cucina europea ha poche ricette, `sample_n = 30` potrebbe essere troppo alto: prova con un valore più basso se ti dà errore di campione insufficiente.

In [ ]:
# TODO: scegli tra 4 e 5 cucine europee presenti in `recipes` (usa la lista stampata sopra)
european_cuisines = None  # TODO: lista di stringhe, ad esempio ["french", "italian", "german", "spanish_portuguese"]

# TODO: filtra `recipes` per tenere solo quelle cucine
european_recipes = None

# TODO: separa la variabile da prevedere (colonna "cuisine") dalle variabili di ingresso (il resto delle colonne)
cuisines = None
ingredients = None

# TODO: definisci quante ricette di ogni cucina vanno nell'insieme di test
sample_n = None  # ad esempio 15 o 20, a seconda di quante ricette ha la cucina con meno esempi

random.seed(1234)

# TODO: costruisci l'insieme di test campionando `sample_n` ricette da ogni cucina
test_set = None

test_ingredients = None
test_cuisines = None

# TODO: costruisci l'insieme di addestramento con tutto ciò che NON è finito nell'insieme di test
train_set = None
train_ingredients = None
train_cuisines = None

# TODO: crea e addestra un DecisionTreeClassifier con l'insieme di addestramento
euro_tree = None

# TODO: prevedi le cucine dell'insieme di test con .predict()
euro_pred_cuisines = None

print("Cocinas usadas:", european_cuisines)

In [ ]:
# TODO: usa confusion_matrix per confrontare test_cuisines contro euro_pred_cuisines,
# convertila in percentuale per riga, e disegnala con plt.imshow (come nel notebook di teoria)

labels = None  # TODO: np.unique(test_cuisines)
conf_matrix = None  # TODO: confusion_matrix(test_cuisines, euro_pred_cuisines, labels=labels)

# TODO: converti conf_matrix in percentuale per riga e disegnala con plt.imshow,
# mettici titolo, colorbar, tick con i nomi di cucina, e i valori come testo in ogni cella

<details>
<summary>💡 Vedi soluzione</summary>

```python
# Scegliamo 4 cucine europee presenti nel dataset pulito
european_cuisines = ["french", "italian", "german", "spanish_portuguese"]

european_recipes = recipes[recipes.cuisine.isin(european_cuisines)]

cuisines = european_recipes["cuisine"]
ingredients = european_recipes.iloc[:, 1:]

# Usiamo un sample_n moderato nel caso qualche cucina europea abbia poche ricette
sample_n = 20

random.seed(1234)
test_set = european_recipes.groupby("cuisine", group_keys=False).apply(lambda x: x.sample(sample_n))

test_ingredients = test_set.iloc[:, 1:]
test_cuisines = test_set["cuisine"]

test_index = european_recipes.index.isin(test_set.index)
train_set = european_recipes[~test_index]
train_ingredients = train_set.iloc[:, 1:]
train_cuisines = train_set["cuisine"]

euro_tree = tree.DecisionTreeClassifier(max_depth=15)
euro_tree.fit(train_ingredients, train_cuisines)

euro_pred_cuisines = euro_tree.predict(test_ingredients)

print("Cocinas usadas:", european_cuisines)

# Matrice di confusione
labels = np.unique(test_cuisines)
conf_matrix = confusion_matrix(test_cuisines, euro_pred_cuisines, labels=labels)
conf_matrix = (conf_matrix.astype("float") / conf_matrix.sum(axis=1)[:, np.newaxis]) * 100

plt.figure(figsize=(8, 6))
plt.imshow(conf_matrix, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Matriz de confusión — cocinas europeas")
plt.colorbar()
tick_marks = np.arange(len(labels))
plt.xticks(tick_marks, labels)
plt.yticks(tick_marks, labels)

thresh = conf_matrix.max() / 2.0
for i, j in itertools.product(range(conf_matrix.shape[0]), range(conf_matrix.shape[1])):
    plt.text(j, i, format(conf_matrix[i, j], ".2f"),
             horizontalalignment="center",
             color="white" if conf_matrix[i, j] > thresh else "black")

plt.tight_layout()
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()
```

Nota: poiché l'albero si addestra con campionamento casuale, le tue cucine confuse tra loro possono variare rispetto ad altri compagni, ma il pattern generale (alcune cucine europee condividono molti ingredienti, come francese e italiana) dovrebbe reggere.

</details>

## Esercizio 2 — Calcola l'accuracy globale a mano

Ora che hai `test_cuisines` (le cucine reali) e `euro_pred_cuisines` (le cucine previste) dell'esercizio 1, calcola l'accuracy globale del tuo albero.

**Suggerimenti:**
- `accuracy_score(y_real, y_predicho)` di `sklearn.metrics` restituisce la proporzione di previsioni corrette sul totale.
- Confronta quel numero con l'accuracy dell'albero di ricette asiatiche/indiana del notebook di teoria (circa 60-77% di successi per cucina). Il tuo albero di cucine europee azzecca di più, di meno o in modo simile?

In [ ]:
# TODO: calcola l'accuracy globale confrontando test_cuisines contro euro_pred_cuisines
accuracy_global = None

print(f"Accuracy global del árbol de cocinas europeas: {accuracy_global:.2%}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
# accuracy_score confronta le previsioni con i valori reali e restituisce
# la proporzione di successi (tra 0 e 1)
accuracy_global = accuracy_score(test_cuisines, euro_pred_cuisines)

print(f"Accuracy global del árbol de cocinas europeas: {accuracy_global:.2%}")
```

Confronta questo numero con l'accuracy globale calcolata nel notebook di teoria per l'albero di cucine asiatiche e indiana. Non c'è un numero "corretto" unico: ciò che conta è che tu capisca che questo è lo stesso tipo di metrica vista nella tabella del caso CHF (colonna "Accuracy globale"), e che un solo numero globale non ti dice nulla su quali cucine specifiche si confondono tra loro — per questo serve la matrice di confusione dell'esercizio 1.

</details>

## Per chiudere

Se hai completato i due esercizi, ora puoi:

- Addestrare un `DecisionTreeClassifier` di scikit-learn su qualsiasi sottoinsieme di classi di un dataset tabulare.
- Dividere dati in addestramento e test, prevedere con `.predict()`, e leggere una matrice di confusione reale.
- Calcolare l'accuracy globale di un modello con `accuracy_score` e capire perché quel singolo numero non racconta mai tutta la storia.

Nella prossima lezione vedrai cosa succede dopo aver avuto un modello valutato e accettato: come si distribuisce nel mondo reale e come il feedback lo migliora nel tempo.